# 🎯 HUẤN LUYỆN YOLO11 ĐA LỚP (8 NHÓM BỆNH HỌC) PHÁT HIỆN & CHẨN ĐOÁN U BUỒNG TRỨNG (MMOTU)
### *Fine-grained Multi-class Ovarian Tumor Detection from Ultrasound Images using YOLO11*

Đề tài NCKH: **Ứng dụng học sâu cho phát hiện khối u buồng trứng từ ảnh siêu âm**.
Notebook này được thiết kế và tối ưu để chạy trực tiếp trên **Google Colab (GPU NVIDIA T4 miễn phí)**, huấn luyện mô hình YOLO11 Nano phân loại chi tiết **8 nhóm bệnh học** của buồng trứng:

| Class ID | Tên tiếng Anh | Tên tiếng Việt | Bản chất lâm sàng |
| :---: | :--- | :--- | :--- |
| **0** | **Chocolate cyst** | U nang sô-cô-la (Lạc nội mạc tử cung) | Lành tính (*Benign*) |
| **1** | **Serous cystadenoma** | U nang thanh dịch | Lành tính (*Benign*) |
| **2** | **Teratoma** | U quái buồng trứng (Nang bì) | Lành tính (*Benign*) |
| **3** | **Theca cell tumor** | U tế bào vỏ | Lành tính (*Benign*) |
| **4** | **Simple cyst** | U nang đơn giản (Nang cơ năng) | Lành tính (*Benign*) |
| **5** | **Normal ovary** | Buồng trứng bình thường (Mẫu đối chứng âm) | Bình thường (*Normal*) |
| **6** | **Mucinous cystadenoma** | U nang nhầy | Lành tính (*Benign*) |
| **7** | **High-grade serous** | Ung thư biểu mô thanh dịch ác tính cao | **Ác tính (*Malignant*)** |


## 1. Thiết lập Môi trường & Kiểm tra GPU (NVIDIA T4)
> **Lưu ý quan trọng**: Trước khi bắt đầu, hãy đảm bảo Colab đã bật GPU:
> Menu **Thời lượng chạy (Runtime)** -> **Thay đổi loại thời lượng chạy (Change runtime type)** -> Chọn **T4 GPU** -> Bấm **Lưu (Save)**.


In [ ]:
# 1. Cài đặt các thư viện cần thiết
!pip install -q ultralytics opencv-python-headless matplotlib seaborn pandas

# 2. Kiểm tra thông tin GPU NVIDIA
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU!")


## 2. Tải Dữ liệu & Mã nguồn Đề tài từ GitHub
Dữ liệu MMOTU và toàn bộ mã nguồn tiền xử lý đã có sẵn trên GitHub repository của đề tài. Cell dưới đây sẽ tự động clone mã nguồn và dữ liệu về môi trường Colab.


In [ ]:
from google.colab import drive
import os, shutil
from pathlib import Path

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Clone repository chứa mã nguồn đề tài
REPO_URL = 'https://github.com/son7nui/NCKH-U-buong-trung.git'
PROJECT_DIR = Path('/content/NCKH')

if not PROJECT_DIR.exists():
    print('🚀 Đang clone repository từ GitHub về Colab...')
    !git clone $REPO_URL $PROJECT_DIR
else:
    print('🔄 Đang cập nhật code mới nhất từ GitHub...')
    !cd $PROJECT_DIR && git pull origin main

# 3. Tự động liên kết dữ liệu MMOTU từ Drive của bạn nếu có
drive_candidates = [
    Path('/content/drive/MyDrive/dataset/MMOTU'),
    Path('/content/drive/MyDrive/MMOTU'),
    Path('/content/drive/MyDrive/dataset')
]
for cand in drive_candidates:
    if cand.exists() and (cand / 'OTU_2d').exists():
        print(f'🎯 Đã tìm thấy dataset trên Drive: {cand}')
        shutil.copytree(cand / 'OTU_2d', PROJECT_DIR / 'MMOTU' / 'OTU_2d', dirs_exist_ok=True)
        break

%cd $PROJECT_DIR
print(f'Thư mục làm việc hiện tại: {os.getcwd()}')


## 3. Tiền xử lý & Chuẩn hóa Dữ liệu YOLO (8 Lớp)
Script `src/prepare_yolo_dataset.py` sẽ tự động:
1. Đọc 1.469 ảnh siêu âm từ `MMOTU/OTU_2d/images`.
2. Trích xuất chính xác tọa độ bounding box từ mặt nạ `_binary.PNG`.
3. Gán nhãn tương ứng từ `0` đến `7` theo chẩn đoán bác sĩ.
4. Đưa 180 ảnh buồng trứng bình thường (Lớp 5) thành mẫu âm bản (background không có box) để chống báo động giả.
5. Tạo file cấu hình `dataset_yolo/dataset.yaml`.


In [ ]:
# Chạy tiền xử lý dữ liệu sang chuẩn YOLO
!python src/prepare_yolo_dataset.py


## 4. Huấn luyện Mô hình YOLO11n Đa Lớp (50 Epochs trên GPU T4)
- **Kiến trúc**: `yolo11n.pt` (YOLO11 Nano - 2.59M tham số, tối ưu cho xử lý ảnh y tế thời gian thực).
- **Hàm mất mát & Tối ưu**: `AdamW` optimizer, learning rate khởi tạo `0.001`, scheduler `cosine lr`.
- **Kích thước ảnh**: `imgsz=640` (chuẩn độ phân giải cao).
- **Thời gian huấn luyện dự kiến**: Khoảng 15 - 20 phút trên GPU T4.


In [ ]:
from ultralytics import YOLO

# Khởi tạo mô hình YOLO11n
model = YOLO('yolo11n.pt')

# Bắt đầu quá trình huấn luyện
results = model.train(
    data='/content/NCKH/dataset_yolo/dataset.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=20,
    save=True,
    plots=True,
    project='runs_colab/detect',
    name='ovarian_tumor_yolo11n_8cls',
    verbose=True
)

print("🎉 Huấn luyện thành công 50 epochs!")


## 5. Đánh giá Chuyên sâu trên Tập Kiểm định (Validation Evaluation)
Tính toán các chỉ số chuyên môn: **mAP@50**, **mAP@50-95**, **Precision**, **Recall** cho toàn bộ 8 nhóm u.


In [ ]:
# Đánh giá trên tập val
val_results = model.val(
    data='/content/NCKH/dataset_yolo/dataset.yaml',
    split='val',
    project='runs_colab/detect',
    name='val_evaluation'
)

print(f"\n📊 KẾT QUẢ ĐÁNH GIÁ CHUNG:")
print(f"mAP@50 toàn cục: {val_results.box.map50 * 100:.2f}%")
print(f"mAP@50-95 toàn cục: {val_results.box.map * 100:.2f}%")
print(f"Precision trung bình: {val_results.box.mp * 100:.2f}%")
print(f"Recall trung bình: {val_results.box.mr * 100:.2f}%")


## 6. Hiển thị Biểu đồ Kết quả & Ma trận Nhầm lẫn (Confusion Matrix)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

exp_dir = Path('/content/NCKH/runs_colab/detect/ovarian_tumor_yolo11n_8cls')

charts = [
    ('Tiến trình Huấn luyện (Loss & mAP)', exp_dir / 'results.png'),
    ('Ma trận Nhầm lẫn Chuẩn hóa', exp_dir / 'confusion_matrix_normalized.png'),
    ('Đường cong Precision-Recall', exp_dir / 'BoxPR_curve.png'),
    ('Đường cong F1-Score', exp_dir / 'BoxF1_curve.png')
]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.flatten()

for idx, (title, path) in enumerate(charts):
    if path.exists():
        img = mpimg.imread(str(path))
        axes[idx].imshow(img)
        axes[idx].set_title(title, fontsize=14, fontweight='bold')
        axes[idx].axis('off')
    else:
        axes[idx].set_title(f"{title} (Không tìm thấy)", fontsize=12)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()


## 7. Trực quan hóa Mẫu Đối đầu (Ground Truth vs Model Prediction)
Chạy suy luận trên các ca bệnh kiểm định và xuất ảnh so sánh song song giữa nhãn của bác sĩ chuyên khoa (xanh lá) và dự đoán của mô hình YOLO11 (đỏ/xanh).


In [ ]:
# Chạy script đánh giá trực quan hóa
!python src/evaluate_and_visualize.py --weights runs_colab/detect/ovarian_tumor_yolo11n_8cls/weights/best.pt --samples 8

# Hiển thị 4 ca bệnh so sánh mẫu
import glob

compare_images = sorted(glob.glob('/content/NCKH/results/visualizations/compare_*.jpg'))[:4]

if compare_images:
    fig, axes = plt.subplots(len(compare_images), 1, figsize=(14, 5 * len(compare_images)))
    if len(compare_images) == 1:
        axes = [axes]
    for ax, img_p in zip(axes, compare_images):
        img = mpimg.imread(img_p)
        ax.imshow(img)
        ax.set_title(f"Ca bệnh: {Path(img_p).name} | Bên trái: Ground Truth (Bác sĩ) vs Bên phải: YOLO11 Prediction", fontsize=12)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## 8. Tự động Sao lưu Trọng số và Kết quả về Google Drive
> Chạy cell này để lưu file trọng số mô hình tốt nhất (`best.pt`) và toàn bộ biểu đồ sang thư mục Google Drive của bạn, đảm bảo kết quả không bị mất khi đóng trình duyệt hoặc tắt Google Colab.


In [ ]:
from google.colab import drive

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Tạo thư mục sao lưu trên Drive
backup_dir = Path('/content/drive/MyDrive/NCKH_Ovarian_Tumor_YOLO11_8Cls')
backup_dir.mkdir(parents=True, exist_ok=True)

# 3. Sao chép trọng số và biểu đồ
weights_src = exp_dir / 'weights' / 'best.pt'
if weights_src.exists():
    shutil.copy2(weights_src, backup_dir / 'best_yolo11n_8cls.pt')
    print(f"✅ Đã sao lưu trọng số vào: {backup_dir / 'best_yolo11n_8cls.pt'}")

# Sao chép các biểu đồ kết quả
for chart in exp_dir.glob('*.png'):
    shutil.copy2(chart, backup_dir / chart.name)

# Sao chép file csv kết quả
csv_src = exp_dir / 'results.csv'
if csv_src.exists():
    shutil.copy2(csv_src, backup_dir / 'results.csv')

print(f"🎉 Hoàn tất sao lưu toàn bộ kết quả thực nghiệm về Google Drive tại thư mục: NCKH_Ovarian_Tumor_YOLO11_8Cls")
